# Notebook 01: Static malware detection — PE headers

**Class 04_01.** Windows PE (Portable Executable) files carry a header with dozens of metadata fields: sizes,
linker and OS versions, section layout, entry point. A scanner can classify a file from these fields alone, without
running it or disassembling it. We build the detector, measure how the benign/malware imbalance hurts it, and
leave the field set wide open for the evasion class (04_03).

In [ ]:
import sys

import numpy as np
import polars as pl

sys.path[:0] = [
    ".",
    "..",
    "../../src",
    "../../../src",
]  # this folder, the course folder (malwlib) and the generic libraries
import aasplot as ap
import malwlib as mw

ap.setup("04")
X, y, cols = mw.load_pe()
print(f"PE header data: {X.shape[0]:,} files x {X.shape[1]} features, {y.mean():.1%} malware")

## The data

Each row is one file; the `Name` column is a hash (`VirusShare_*` for malware). 77 of the 79 columns are numeric
header fields; three quarters of the files are malware, which is nothing like the <1% of a real scanner queue —
we exploit that gap in the imbalance experiment below.

In [ ]:
raw = pl.read_csv(mw.DATA / "benign_malicious_pe_files_train.csv.zst", infer_schema_length=0)
print(raw.select(pl.col("Name").str.starts_with("VirusShare").value_counts(), pl.col("Malware").value_counts()))

## Baseline detector

A random forest on the 77 fields. At this ratio the task is nearly solved — remember this number when the
attack arrives.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
rf = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr, ytr)
p = rf.predict_proba(Xte)[:, 1]
mw.eval_bin(yte, p, "RF PE baseline: ")

## Class imbalance: benign downsampled

A scanner sees mostly benign files. Downsampling the benign training class to 20% and 5% simulates what happens
when the detector is trained on a queue with realistic rates: recall on malware stays high but false positives
(benign files flagged) degrade — the metric that costs the most in practice.

In [ ]:
rng = np.random.default_rng(0)
idx_m, idx_b = np.where(ytr == 1)[0], np.where(ytr == 0)[0]
rows = []
for frac in [1.0, 0.2, 0.05, 0.01]:
    nb = max(50, int(len(idx_b) * frac))
    keep = rng.permutation(np.concatenate([idx_m, rng.choice(idx_b, nb, replace=False)]))
    rf2 = RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1).fit(Xtr[keep], ytr[keep])
    p2 = rf2.predict_proba(Xte)[:, 1]
    e = mw.eval_bin(yte, p2, f"benign frac={frac:4.2f} (n={nb:4d}): ")
    rows.append({"benign kept": nb, "AUC": e["auc"], "AUPRC": e["auprc"], "F1@.5": e["f1"]})

In [ ]:
f, ax = ap.fig("col", 1.9)
fr = [1.0, 0.2, 0.05, 0.01]
ax.plot(fr, [r["AUC"] for r in rows], "-o", color=ap.BLUE, label="AUC")
ax.plot(fr, [r["F1@.5"] for r in rows], "-o", color=ap.RED, label="F1@.5")
ax.set_xscale("log")
ax.set_xlabel("benign training fraction")
ax.invert_xaxis()
ax.legend()
ap.save(f, "d04_imbalance")

## Threshold on the base rate

With a 1% true malware rate the operating point moves: precision collapses even at the same AUC, and the
threshold that maximises F1 shifts. This is the practical reason the imbalance experiment matters.

In [ ]:
p2 = rf2.predict_proba(Xte)[:, 1]
thr = np.linspace(0.05, 0.95, 19)
flagged = p2 > thr[:, None]  # one row per threshold
prec = np.array([np.mean(yte[r]) for r in flagged])  # P(malware | flagged)
rec = np.array([yte[r].mean() for r in flagged])
f1 = 2 * prec * rec / np.maximum(1e-9, prec + rec)
print(
    f"best threshold {thr[f1.argmax()]:.2f} -> F1 {f1.max():.3f} (precision {prec[f1.argmax()]:.3f}, recall {rec[f1.argmax()]:.3f})"
)

In [ ]:
f, ax = ap.fig("col", 1.9)
ax.plot(thr, prec, "-o", color=ap.BLUE, label="precision")
ax.plot(thr, rec, "-o", color=ap.GREEN, label="recall")
ax.plot(thr, f1, "-o", color=ap.RED, label="F1")
ax.set_xlabel("threshold")
ax.legend()
ap.save(f, "d04_threshold")

## What the model looks at

The most important fields are version and size indicators — exactly the fields a packer can rewrite freely
(class 04_03 rewrites them and the detector folds).

In [ ]:
imp = rf.feature_importances_
top = np.argsort(imp)[-10:][::-1]
for i in top[:8]:
    print(f"{cols[i]:34s} {imp[i]:.3f}")

In [ ]:
f, ax = ap.fig("col", 2.2)
ax.barh(np.arange(8), imp[top[:8]][::-1], color=ap.BLUE)
ax.set_yticks(np.arange(8), [c[:24] for c in np.array(cols)[top[:8]]][::-1])
ax.set_xlabel("importance")
ap.save(f, "d04_importance")

## Takeaways

- A 77-field PE header model reaches AUC ≈ .999 — static header detection is *solved* on this benchmark, which
  also makes it a fragile teaching example: the fields are freely malleable.
- Benign downsampling keeps malware recall high but the precision/FP side degrades; the F1-optimal threshold
  moves with the base rate, so report operating points, not AUC alone.
- Top features (`MajorLinkerVersion`, OS/subsystem versions, sizes) are exactly what class 04_03 rewrites.